# Kuku redo

## Step 1: read one episode

Find every `Start: (...)` line. The line above it says who, the line below it says what.

In [ ]:
import re

path = "/Users/armaanjotsingh/Desktop/projectPlacement/I Married a Broken Soldier. He was a Duke/episode-002-Episode 1671704.txt"
text = open(path, encoding="utf-8").read()
rows = text.split("\n")

count = 0
for i, row in enumerate(rows):
    m = re.match(r"Start: \((\d+:\d+\.\d+)\) \| End: \((\d+:\d+\.\d+)\)", row.strip())
    if not m:
        continue
    who = rows[i - 1].strip()
    start = m.group(1)
    end = m.group(2)
    what = rows[i + 1].strip()
    print(who, start, end, what)
    count += 1

print(count, "timed lines")

## Step 2: one episode → one dict

The header (number, logline) plus a list of lines, each line a dict.

In [ ]:
def to_seconds(t):
    mins, secs = t.split(":")
    return int(mins) * 60 + float(secs)

def parse_episode(path):
    text = open(path, encoding="utf-8").read()
    rows = text.split("\n")
    number = re.search(r"EPISODE NUMBER:\s*Episode\s*(\d+)", text)
    logline = re.search(r"EPISODE LOGLINE:\s*(.*)", text)
    lines = []
    for i, row in enumerate(rows):
        m = re.match(r"Start: \((\d+:\d+\.\d+)\) \| End: \((\d+:\d+\.\d+)\)", row.strip())
        if not m:
            continue
        head = rows[i - 1].strip().strip("*").rstrip(":")
        speaker = head.split("(")[0].strip()
        emotion = ", ".join(re.findall(r"\((.*?)\)", head))
        if speaker in ("CAMERA", "ACTION", "SOUND FX"):
            kind = speaker
        else:
            kind = "DIALOGUE"
        lines.append({
            "start": to_seconds(m.group(1)),
            "end": to_seconds(m.group(2)),
            "kind": kind,
            "speaker": speaker,
            "emotion": emotion,
            "text": rows[i + 1].strip() if i + 1 < len(rows) else "",
        })
    if not lines:
        raise ValueError(f"No timed lines found in {path}. Check the file format.")
    return {
        "number": int(number.group(1)) if number else None,
        "logline": logline.group(1).strip() if logline else "",
        "length": max(line["end"] for line in lines),
        "lines": lines,
    }

ep = parse_episode(path)
print("episode", ep["number"], "|", len(ep["lines"]), "lines |", ep["length"], "s")
print(ep["logline"])

## Step 3: read all 50 episodes

In [ ]:
import os

FOLDER = "/Users/armaanjotsingh/Desktop/projectPlacement/I Married a Broken Soldier. He was a Duke"

episodes = []
for name in sorted(os.listdir(FOLDER)):
    if name.endswith(".txt"):
        episodes.append(parse_episode(os.path.join(FOLDER, name)))

print(len(episodes), "episodes,", sum(len(e["lines"]) for e in episodes), "lines")

## Step 4: the short version the AI will read

Drop CAMERA lines (half the text), number the rest L1, L2, ...

In [ ]:
def story_lines(ep):
    return [line for line in ep["lines"] if line["kind"] != "CAMERA"]

def shrink(ep):
    out = []
    for n, line in enumerate(story_lines(ep), start=1):
        out.append(f"L{n} [{line['start']:.1f}-{line['end']:.1f}] {line['speaker']}: {line['text']}")
    return "\n".join(out)

print(shrink(episodes[1])[:400])

## Step 5: check a clip by line numbers

Pick a hook line and a cliffhanger line; code works out the length.

In [ ]:
def check_clip(ep, from_line, to_line):
    story = story_lines(ep)
    if not 1 <= from_line <= to_line <= len(story):
        print(f"ep {ep['number']}: L{from_line}-L{to_line} -> bad line numbers (this episode has L1-L{len(story)})")
        return
    start = story[from_line - 1]["start"]
    end = story[to_line - 1]["end"]
    duration = round(end - start, 1)
    ok = 20 <= duration <= 60
    print(f"ep {ep['number']}: {start}s to {end}s = {duration}s ->", "OK" if ok else "breaks the 20-60s rule")
    print("  hook:", story[from_line - 1]["text"])
    print("  cliffhanger:", story[to_line - 1]["text"])

ep2 = episodes[1]
check_clip(ep2, 1, 14)

## Step 6: ask the AI for promos (one episode)

The AI only picks line numbers. Code (`check_clip`) does the time maths. Needs Ollama running: `ollama serve`.

In [ ]:
from openai import OpenAI
from pydantic import BaseModel

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL = "qwen2.5:7b"

class Promo(BaseModel):
    from_line: int
    to_line: int
    why: str
    score: int

class Picks(BaseModel):
    promos: list[Promo]

PROMPT = """You pick promo clips from a short vertical drama, to pull new viewers in on TikTok.
Lines are numbered L1, L2, ... with [start-end] in seconds. Find up to 3 promos, each from_line to to_line:
- from_line is the HOOK: a shocking line that stops a stranger scrolling in 3 seconds.
- to_line is the CLIFFHANGER: cut before anything is resolved.
- Length = end of to_line minus start of from_line. It MUST be 20 to 60 seconds.
- A stranger must follow it with no context. score is 1-10, be strict.
Reply with JSON only: {"promos": [{"from_line": 1, "to_line": 14, "why": "one line", "score": 7}]}"""

def ask_ai(ep):
    reply = client.chat.completions.create(
        model=MODEL,
        temperature=0.2,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": PROMPT},
                  {"role": "user", "content": f"Episode {ep['number']}: {ep['logline']}\n\n{shrink(ep)}"}],
    )
    return Picks.model_validate_json(reply.choices[0].message.content).promos

for p in ask_ai(episodes[1]):
    check_clip(episodes[1], p.from_line, p.to_line)
    print("  why:", p.why, "| score", p.score)

## Step 6b: let the AI fix its own mistakes (one retry)

Code lists exactly what broke the rules and sends it back. This replaces `ask_ai`, so step 7 uses the retry automatically.

In [ ]:
def what_is_wrong(ep, picks):
    story = story_lines(ep)
    problems = []
    for i, p in enumerate(picks, start=1):
        if not 1 <= p.from_line <= p.to_line <= len(story):
            problems.append(f"Promo {i}: line numbers must be between L1 and L{len(story)}.")
            continue
        duration = story[p.to_line - 1]["end"] - story[p.from_line - 1]["start"]
        if not 20 <= duration <= 60:
            problems.append(f"Promo {i} is {duration:.1f} seconds long. It must be 20 to 60 seconds.")
    return "\n".join(problems)

def ask_ai(ep):
    messages = [{"role": "system", "content": PROMPT},
                {"role": "user", "content": f"Episode {ep['number']}: {ep['logline']}\n\n{shrink(ep)}"}]
    for attempt in range(2):
        reply = client.chat.completions.create(model=MODEL, temperature=0.2,
                                               response_format={"type": "json_object"}, messages=messages)
        raw = reply.choices[0].message.content
        picks = Picks.model_validate_json(raw).promos
        problems = what_is_wrong(ep, picks)
        if not problems or attempt == 1:
            break
        print(f"  ep {ep['number']} retry because:", problems.replace("\n", " | "))
        messages.append({"role": "assistant", "content": raw})
        messages.append({"role": "user", "content": problems + "\nFix these and send the full JSON again."})
    return picks

## Step 7: run on episodes, keep what passes, rank

Starts with 5 episodes. Change `episodes[:5]` to `episodes` for all 50 (about 15 to 40 min).

In [ ]:
import time
import pandas as pd

def clip_row(ep, p):
    story = story_lines(ep)
    if not 1 <= p.from_line <= p.to_line <= len(story):
        return None
    start = story[p.from_line - 1]["start"]
    end = story[p.to_line - 1]["end"]
    return {"episode": ep["number"], "start": start, "end": end, "duration": round(end - start, 1),
            "hook": story[p.from_line - 1]["text"], "cliffhanger": story[p.to_line - 1]["text"],
            "why": p.why, "score": p.score}

rows = []
for ep in episodes[:5]:
    try:
        picks = ask_ai(ep)
    except Exception as e:
        print(f"ep {ep['number']}: bad reply, skipped ({e.__class__.__name__})")
        continue
    good = 0
    for p in picks:
        row = clip_row(ep, p)
        if row and 20 <= row["duration"] <= 60:
            rows.append(row)
            good += 1
    print(f"ep {ep['number']}: {len(picks)} picks, {good} pass the rules")
    time.sleep(15)

ranked = pd.DataFrame(rows).sort_values("score", ascending=False)
ranked